# Python Pandas Data Cleanup Assignment
**Author:** Student
**Task:** Load a raw CSV, clean null entries, normalize columns, and parse data ranges professionally.

## Step 1: Environment Setup & Data Loading
In this step, we import the necessary libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`) and load the raw dataset. We also generate a realistic flawed dataset to simulate real-world data issues like missing values, inconsistent casing, and unclean strings.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Professional styling for visualizations
sns.set_theme(style="whitegrid")
warnings.filterwarnings('ignore')

print("=== Step 1: Loading Raw Data ===")

# Simulating a flawed raw dataset (In practice, use: df = pd.read_csv('your_file.csv'))
raw_data = {
    'Employee_ID': [101, 102, np.nan, 104, 105, 106],
    'Name': [' Shahbaz Khan ', 'Amna Ali', 'mubashir', 'Zainab Bibi', '  Hamza ', 'Ali Khan'],
    'Salary': ['$5,000', '6000', 'not disclosed', '$4,500', np.nan, '7,200'],
    'Joining_Date': ['2023-01-15', '2022/11/20', '12-05-2021', '2024-02-01', '2023-07-19', np.nan],
    'Experience_Range': ['1-3 Years', '5-8 Years', '0-1 Years', '3-5 Years', '5-8 Years', '1-3 Years']
}

df = pd.DataFrame(raw_data)
print("\nRaw Data Snapshot:")
display(df.head())
print("\nMissing Values Count before cleaning:")
print(df.isnull().sum())

## Step 2: Handling Null & Missing Entries
Instead of blindly dropping rows, we handle missing data carefully:
- Critical identification columns like `Employee_ID` cannot have missing values, so we drop rows missing them.
- Missing textual names are imputed with a logical placeholder (`Unknown`).
- Missing numerical values will be handled strategically after cleaning symbols in the next step.

In [ ]:
print("=== Step 2: Cleaning Null Entries ===")

# 1. Dropping rows where critical identifier (Employee_ID) is missing
df = df.dropna(subset=['Employee_ID'])
df['Employee_ID'] = df['Employee_ID'].astype(int)

# 2. Filling missing text values with a placeholder
df['Name'] = df['Name'].fillna('Unknown')

print("\nNull values after initial critical cleaning:")
print(df.isnull().sum())

## Step 3: Data Normalization & Formatting
Here we format and standardize the dataset columns:
1. Strip leading/trailing whitespaces and convert names to proper **Title Case**.
2. Remove currency symbols (`$`) and commas (`,`) from `Salary`, coerce invalid string inputs to `NaN`, and intelligently impute missing salaries using the **median** value of the column (industry best practice).
3. Standarize conflicting date formats into a unified `YYYY-MM-DD` structure.

In [ ]:
print("=== Step 3: Normalizing Columns ===")

# 1. String Normalization
df['Name'] = df['Name'].str.strip().str.title()

# 2. Numeric Normalization (Cleaning Currency & String Data)
df['Salary'] = df['Salary'].astype(str).str.replace('$', '').str.replace(',', '')
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

# Imputing missing Salary values with the median salary
median_salary = df['Salary'].median()
df['Salary'] = df['Salary'].fillna(median_salary)

# 3. Date Normalization (Handling mixed/conflicting date strings)
df['Joining_Date'] = pd.to_datetime(df['Joining_Date'], errors='coerce', format='mixed')
# Filling any missing dates with the mode (most frequent date)
df['Joining_Date'] = df['Joining_Date'].fillna(df['Joining_Date'].mode()[0])

print("\nNormalized and Cleaned Dataset:")
display(df)

## Step 4: Parsing Data Ranges (Feature Engineering)
To capture insights from categorical text ranges (e.g., `1-3 Years`), we use Regular Expressions (`str.extract`) to parse out explicit `Min_Exp` and `Max_Exp` numeric metrics, creating a valuable engineered feature: `Avg_Exp`.

In [ ]:
print("=== Step 4: Parsing Data Ranges ===")

# Extracting minimum and maximum boundaries using Regular Expressions
df[['Min_Exp', 'Max_Exp']] = df['Experience_Range'].str.extract(r'(\d+)-(\d+)')

# Convert types to integer
df['Min_Exp'] = df['Min_Exp'].astype(int)
df['Max_Exp'] = df['Max_Exp'].astype(int)

# Feature Engineering: Calculate Average Experience
df['Avg_Exp'] = (df['Min_Exp'] + df['Max_Exp']) / 2

# Dropping the redundant unparsed column
df = df.drop(columns=['Experience_Range'])

print("\nFinal Processed Data with Parsed Ranges:")
display(df)

## Step 5: Visualizing Cleaned Insights & Export
Finally, we visualize trends within our clean dataset to demonstrate analytical value, and export the output to a standard CSV file format.

In [ ]:
print("=== Step 5: Visualizing Cleaned Insights ===")

plt.figure(figsize=(12, 5))

# Plot 1: Employee Salaries
plt.subplot(1, 2, 1)
sns.barplot(x='Name', y='Salary', data=df, palette='Blues_r')
plt.title('Normalized Salary Distribution per Employee')
plt.xticks(rotation=45)

# Plot 2: Relationship Between Experience and Salary
plt.subplot(1, 2, 2)
sns.scatterplot(x='Avg_Exp', y='Salary', data=df, s=150, color='crimson', edgecolor='black')
plt.title('Salary Trend vs. Average Experience')
plt.xlabel('Average Experience (Years)')
plt.ylabel('Salary ($)')

plt.tight_layout()
plt.show()

# Exporting to production ready CSV
df.to_csv('cleaned_employee_data.csv', index=False)
print("\n[SUCCESS] Pipeline executed perfectly. Cleaned data saved as 'cleaned_employee_data.csv'!")